# 02 - Baseline Results Analysis

Load saved results from all baseline runs and produce comparison tables and figures.
No training here - run baseline scripts first, then use this notebook to analyse.

**Prerequisite:** Run these scripts first:
```bash
python -m baselines.random_walk
python -m baselines.vanilla_lstm
# ... other baselines as they are implemented
```

In [ ]:
import sys, json, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
from IPython.display import Image, display, Markdown

from utils.logger import save_method_table, save_comparison_table, print_method_summary
from utils.visualizer import plot_rmse_comparison, plot_rmse_boxplots, plot_predictions, plot_wilcoxon_heatmap
from utils.metrics import wilcoxon_test

RESULTS_DIR = '../results/baselines'
TICKERS = [
    'JKH.N0000', 'COMB.N0000', 'DIAL.N0000', 'HNB.N0000',  'LOLC.N0000',
    'SAMP.N0000', 'NTB.N0000', 'HHL.N0000',  'DIST.N0000', 'HAYL.N0000',
]

METHOD_FILES = {
    'random_walk':   'random_walk.json',
    'vanilla_lstm':  'vanilla_lstm.json',
    'grid_search':   'grid_search.json',
    'pso_lstm':      'pso_lstm.json',
    'ga_lstm':       'ga_lstm.json',
    'gwo_lstm':      'gwo_lstm.json',
    'woa_lstm':      'woa_lstm.json',
    'sequential_fa': 'sequential_fa.json',
}

# Load whatever exists
all_results = {}
for method, fname in METHOD_FILES.items():
    path = os.path.join(RESULTS_DIR, fname)
    if os.path.exists(path):
        with open(path) as f:
            all_results[method] = json.load(f)
        print(f'Loaded: {method}')
    else:
        print(f'Missing (not run yet): {method}')

## 1. Per-Method Summary Tables

In [ ]:
for method, results in all_results.items():
    print_method_summary(method, results)
    save_method_table(method, results)

## 2. Full Comparison Table

In [ ]:
# Add MOMFA if available
momfa_path = '../results/momfa/momfa.json'
if os.path.exists(momfa_path):
    with open(momfa_path) as f:
        all_results['momfa'] = json.load(f)
    print('MOMFA results loaded')
else:
    print('MOMFA not run yet - comparison table will exclude it')

paths = save_comparison_table(all_results)
if paths:
    with open(paths['md']) as f:
        display(Markdown(f.read()))

## 3. RMSE Bar Comparison

In [ ]:
methods_order = list(all_results.keys())
path = plot_rmse_comparison(all_results, methods_order=methods_order)
display(Image(path))

## 4. 30-Run RMSE Box Plots

Shows statistical variability across 30 independent runs.
Supports the Wilcoxon signed-rank test argument.

In [ ]:
path = plot_rmse_boxplots(all_results, methods_order=methods_order)
display(Image(path))

## 5. Wilcoxon Signed-Rank Test

Tests whether MOMFA is significantly better than each baseline (p < 0.05).
Requires both MOMFA and baseline results with 30 runs each.

In [ ]:
if 'momfa' not in all_results:
    print('MOMFA results not available yet. Run momfa experiments first.')
else:
    p_values = {}
    print(f'{"Method":<20} {"Ticker":<10} {"p-value":<10} {"Significant?"}')
    print('-' * 55)
    
    for method in all_results:
        if method == 'momfa':
            continue
        p_values[method] = {}
        for ticker in TICKERS:
            if ticker not in all_results[method] or ticker not in all_results['momfa']:
                continue
            a = np.array(all_results['momfa'][ticker]['rmse_runs'])
            b = np.array(all_results[method][ticker]['rmse_runs'])
            _, p = wilcoxon_test(a, b)
            p_values[method][ticker] = p
            sig = '✓ YES' if p < 0.05 else '  no'
            print(f'{method:<20} {ticker.replace(".N0000",""):<10} {p:<10.4f} {sig}')
    
    path = plot_wilcoxon_heatmap(p_values)
    display(Image(path))

## 6. Predictions vs Actual - Best Seed per Ticker

In [ ]:
if 'vanilla_lstm' in all_results:
    for ticker in TICKERS:
        r = all_results['vanilla_lstm'].get(ticker, {})
        preds = r.get('best_val_preds', [])
        true  = r.get('best_val_true',  [])
        if preds and true:
            path = plot_predictions(
                np.array(true), np.array(preds),
                ticker=ticker, fold=4
            )
            display(Image(path))
else:
    print('Vanilla LSTM results not available yet.')

## 7. Per-Ticker RMSE Heatmap - All Methods

In [ ]:
import seaborn as sns

ticker_labels = [t.replace('.N0000', '') for t in TICKERS]
available_methods = list(all_results.keys())

matrix = []
for method in available_methods:
    row = []
    for ticker in TICKERS:
        if ticker in all_results[method]:
            row.append(np.mean(all_results[method][ticker]['rmse_runs']))
        else:
            row.append(np.nan)
    matrix.append(row)

method_labels = [
    {'random_walk': 'Random Walk', 'vanilla_lstm': 'Vanilla LSTM',
     'grid_search': 'Grid Search', 'pso_lstm': 'PSO-LSTM',
     'ga_lstm': 'GA-LSTM', 'gwo_lstm': 'GWO-LSTM',
     'woa_lstm': 'WOA-LSTM', 'sequential_fa': 'Sequential FA',
     'momfa': 'MOMFA (ours)'}.get(m, m)
    for m in available_methods
]

fig, ax = plt.subplots(figsize=(12, len(available_methods) * 0.6 + 1.5))
sns.heatmap(matrix, ax=ax,
            xticklabels=ticker_labels,
            yticklabels=method_labels,
            annot=True, fmt='.3f',
            cmap='YlOrRd',
            cbar_kws={'label': 'Mean RMSE (LKR)'})
ax.set_title('Mean RMSE per Method per Ticker')
plt.tight_layout()
plt.show()